# Séance(s) - Finitions

Ce notebook nous amène au dernier bloc du cours de visualisation de données. Jusqu'à présent, nous nous sommes concentrés sur la création de graphiques permettant de visualiser les relations entre nos variables. Il est temps de passer à dernière étape : **les finitions**.

Ces finitions sont importantes car un bon graphique n'est pas seulement techniquement correct, il doit être **lisible et rapidement compréhensible**.

## Objectifs 
- Modifier la taille des polices (titres, axes, légendes)
- Contrôler les échelles et les limites des axes
- Personnaliser les légendes et ordonner les modalités
- Utiliser des couleurs porteuses de sens et/ou des formes (shapes)
- Formater le texte (pourcentages, décimales)
- Ajouter des annotations textuelles et des repères (lignes de référence)
- Utiliser les thèmes globaux d'Altair
- Exporter vos graphiques (png, pdf, svg)

**Rappel** : Nous utilisons **pandas** pour préparer les données et **Altair** pour visualiser.


In [ ]:
# ===========================================
# Installation & Chargement des bibliothèques
# ===========================================
%pip install "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

import pandas as pd
import altair as alt
import warnings

warnings.filterwarnings("ignore")

# Configuration d'Altair
alt.data_transformers.enable("vegafusion")
alt.data_transformers.disable_max_rows()


In [ ]:
# ===========================================
# Chargement de la base de données
# ===========================================
data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df_raw = pd.read_csv(data_url, compression="gzip", low_memory=False)


In [ ]:
# Sélectionner les variables d'intérêt
selected_vars = [
    "V241156",  # thermomètre Harris
    "V241157",  # thermomètre Trump
    "V241221",  # identification partisane (7 points)
    "V241465x", # éducation
    "V241458x", # âge
    "V241004",  # intérêt politique
    "V241043",  # intention de vote
]

df = df_raw[selected_vars].copy()

df.columns = [
    "thermo_harris",
    "thermo_trump",
    "party_id",
    "education",
    "age",
    "pol_int",
    "vote_int"
]

# Calcul de la polarisation affective : différence absolue entre les deux thermomètres
# Une valeur élevée = partisan fort (forte évaluation d'un candidat, faible de l'autre)
# Une valeur faible = partisan modéré (évaluations similaires des deux candidats)
df["aff_pol"] = abs(df["thermo_harris"] - df["thermo_trump"])

# Recodage de l'identification partisane
party_labels = {
    1: "Démocrate", 2: "Républicain",
    3: "Indépendant", 5: "Autre"
}
df["party_id"] = df["party_id"].replace(party_labels)
df

# Recodage de l'intention de vote
vote_labels = {
    1: "Harris",
    2: "Trump",
    3: "Autre",
    4: "Autre",
    5: "Autre",
    6: "Autre",
}
df["vote_int_label"] = df["vote_int"].replace(vote_labels)

# Recodage de l'intérêt politique
pol_int_labels = {
    1: "Toujours",
    2: "La plupart du temps",
    3: "Parfois",
    4: "Rarement",
    5: "Jamais"
}
df["pol_int_label"] = df["pol_int"].replace(pol_int_labels)

# Nettoyage : on garde les valeurs valides
mask = (
    df["thermo_harris"].between(0, 100) &
    df["thermo_trump"].between(0, 100) &
    df["aff_pol"].between(0, 100) &
    df["party_id"].isin(["Démocrate", "Indépendant", "Républicain"]) &    
    df["vote_int"].between(1,3) &
    df["pol_int"].between(1, 5)
)
df_clean = df[mask]
df_clean


## 1. Un graphique de base à améliorer

Commençons par créer un graphique de base montrant la **polarisation affective** (mesurée par l'écart entre les évaluations des deux candidats) selon l'**intérêt politique** et l'**intention de vote**. Nous l'améliorerons étape par étape.

**Concept :** Nous créons un graphique "brut" qui contient bien l'information, mais qui manque de finitions. Il est lisible pour vous, mais peut-être pas pour le grand public.


In [ ]:
# Calcul des moyennes de polarisation affective par intérêt politique et intention de vote
df_means = df_clean.groupby(
    ["pol_int", "pol_int_label", "vote_int_label"], 
    as_index=False
    )["aff_pol"].mean()
df_means["aff_pol"] = df_means["aff_pol"].round(1)

df_means


In [ ]:
base_chart = alt.Chart(df_means).mark_point(size=100).encode(
    x=alt.X("pol_int_label", type="ordinal"),
    y=alt.Y("aff_pol", type="quantitative"),
    color=alt.Color("vote_int_label", type="nominal")
).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt politique et l'intention de vote",
        subtitle=["Plus le score est élevé, plus les émotions envers les candidats sont contrastées.", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=500,
    height=500
)

base_chart


Le graphique montre une première relation entre l'intérêt politique, le vote et la polarisation affective. Chaque point représente le niveau moyen de polarisation affective d'un groupe défini par son intérêt politique (axe X) et son intention de vote (couleur). La hauteur du point indique l'intensité de la polarisation.

Cependant, ce graphique présente de nombreaux problèmes. Nous allons progressivement améliorer la mise en forme.

## 2. Ajuster les polices

Le graphique ci-dessus est fonctionnel, mais les textes (titre, axes, légende) sont un peu petits.

**Concept :** Nous pouvons utiliser la méthode `.configure_*()` pour ajuster globalement l'apparence. `configure_title()` gère le titre, `configure_axis()` gère les axes et `configure_legend()` gère la légende. C'est la méthode la plus rapide pour donner un look aéré.

Pour plus d'infos: https://altair-viz.github.io/user_guide/configuration.html

Sachant que notre graphique de base est enregistré dans l'objet `base_chart` nous pouvons directement améliorer ce dernier.


In [ ]:
chart_fonts = base_chart.configure_title(
    fontSize=22,
    subtitleFontSize=14,
    color="#333333"
).configure_axis(
    labelFontSize=14,
    titleFontSize=16,
    labelAngle=0
).configure_legend(
    titleFontSize=16,
    labelFontSize=14,
    orient='left'  # 'right' par défeaut
)

chart_fonts


### Hack-Time
- Changez la taille de police du titre pour la rendre gigantesque (`fontSize=30`).
- Essayez de changer l'angle de l'axe (`labelAngle=45`). Observez le résultat.
- Modifier la position de la légende


In [ ]:
# Hack-Time 


## 3. Nommer les Axes, légendes et ajuster les limites du cadre

Nous l'avons vu précédemment, les éléments dans altair possèdent un paramètre titre qui nous permet d'ajuster ceux-ci. 

Notez que parfois, les axes ne commencent pas à 0. Lorsque c'est le cas, **il est important de fixer l'échelle** pour une lecture honnête. L'objet `alt.Scale(domain=[min, max])` permet de forcer l'axe à afficher une certaine plage de valeurs. De plus, `alt.Axis(grid=False)` permet d'enlever les lignes de quadrillage pour un style plus épuré.


In [ ]:
chart_scales = alt.Chart(df_means).mark_point(size=150, filled=False).encode(
    x=alt.X(
        "pol_int_label", 
        type="ordinal", 
        title="Intérêt politique"
    ),
    y=alt.Y(
        "aff_pol", type="quantitative",
        title="Polarisation affective",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color(
        "vote_int_label", 
        type="nominal", 
        title="Intention de vote"
    )
).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt politique et l'intention de vote",
        subtitle=["Plus le score est élevé, plus les émotions envers les candidats sont contrastées.", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=450,
    height=450
).configure_axis(
    labelFontSize=12, titleFontSize=20, labelAngle=0,
    # grid=False  # On peut aussi désactiver le cadre mais altair les gère déjà très bien
).configure_legend(
    titleFontSize=14, labelFontSize=11
)

chart_scales


### Hack-Time
- Modifiez `domain=[0, 100]` en `domain=[40, 80]` et observez l'effet de zoom sur la perception des écarts.
- Ajustez les tailles des différents éléments pour mettre en évidence le titre principal par rapport au reste.


In [ ]:
# Hack-Time : votre test


## 4. Ordonner les modalités avec `sort`

Nous remarquons aussi que l'axe X n'est pas trié dans un ordre logique.

Par défaut, Altair trie les catégories par ordre alphabétique. Cependant, cet ordre n'est souvent pas pertinent. 

Nous pouvons trier les catégories manuellement avec une liste.

**Concept :** Le paramètre `sort=` dans l'encodage permet de spécifier l'ordre d'affichage. Nous pouvons utiliser cet éléments de différentes manières :
- avec une **liste** de catégories/modalités dans l'ordre souhaité
- `sort="-y"` pour trier par ordre décroissant de l'axe Y
- `sort="y"` pour trier par ordre croissant


In [ ]:
# Ordre logique pour l'intérêt politique (du moins intéressé au plus intéressé)
pol_int_order = ["Jamais", "Rarement", "Parfois","La plupart du temps", "Toujours"]

chart_sorted = alt.Chart(df_means).mark_point(size=150, filled=True).encode(
    x=alt.X(
        "pol_int_label", 
        type="ordinal",
        title="Intérêt politique",
        sort="y"  # utilisez ici la liste `pol_int_order`
    ),  
    y=alt.Y(
        "aff_pol", type="quantitative",
        title="Polarisation affective",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color(
        "vote_int_label", 
        type="nominal", 
        title="Intention de vote",
    )
).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt politique et l'intention de vote",
        subtitle=["Que pouvvons-nous dire ?", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=400, height=300
).configure_axis(labelAngle=0)

chart_sorted


En triant l'axe X de manière logique, la lecture est plus intuitive. Le tri permet de hiérarchiser visuellement les catégories. Selon votre message, vous choisirez le tri le plus pertinent.

### Hack-Time
- Essayer d'utiliser les options de base `sort="-y"` et `sort="y"`. Quelle différence observez-vous ?
- Essayer d'ordonner la légende pour que la catégorie "Autre" soit en dernier


In [ ]:
# Hack-Time 


## 5. Les Couleurs 

En politique, les couleurs ont un sens conventionnel. Aux États-Unis, les Démocrates sont associés au bleu et les Républicains au rouge. Utiliser des couleurs arbitraires peut porter à confusion.

**Concept :** Imposons nos propres couleurs avec `alt.Scale(domain=..., range=...)`. Le paramètre `range` accepte les noms de couleurs ou des codes hexadécimaux. 


In [ ]:
# Définition des couleurs sémantiques
vote_colors = alt.Scale(
    domain=["Harris", "Trump", "Autre"],
    range=["#2E74C0", "#CB454A", "#A9A9A9"]
)

chart_colors = alt.Chart(df_means).mark_point(size=200, filled=True).encode(
    x=alt.X("pol_int_label", type="ordinal",
            title="Intérêt politique",
            sort=pol_int_order),
    y=alt.Y("aff_pol", type="quantitative",
            title="Polarisation affective",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("vote_int_label", type="nominal",
                   title="Intention de vote",
                   scale=vote_colors),
).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt politique et l'intention de vote",
        subtitle=["Que pouvvons-nous dire ?", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=450,
    height=300
).configure_axis(
    labelFontSize=12, titleFontSize=12, labelAngle=0
)

chart_colors


Les couleurs permettent aussi d'aider la lecture: bleu pour Harris, rouge pour Trump. L'ajout de formes différentes améliore l'accessibilité pour les personnes daltoniennes.

Pour finir, nous pouvons aussi créer un graphiques avec des lignes et le combiner.

In [ ]:
dots = chart_colors = alt.Chart(df_means).mark_point(size=200, filled=True).encode(
    x=alt.X("pol_int_label", type="ordinal",
            title="Intérêt politique",
            sort=pol_int_order),
    y=alt.Y("aff_pol", type="quantitative",
            title="Polarisation affective",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("vote_int_label", type="nominal",
                   title="Intention de vote",
                   scale=vote_colors),
)

line = alt.Chart(df_means).mark_line(size=2).encode(
    x=alt.X("pol_int_label",
            type="ordinal",
            title="Intérêt politique",
            sort=pol_int_order),
    y=alt.Y("aff_pol",
            type="quantitative",
            title="Polarisation affective",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("vote_int_label:N",
                    title="Intention de vote",
                    scale=vote_colors)
)

dots + line


In [ ]:
(dots + line).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt politique et l'intention de vote",
        subtitle=["Que pouvvons-nous dire ?", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=450,
    height=300
).configure_axis(
    labelFontSize=12, titleFontSize=12, labelAngle=0
)

## 6. La fonction `melt()` : passer du format large au format long

Dans le graphique précédent, nous avons une variable (polarisation affective) représentée en fonction de deux variables de regroupement (intérêt politique et vote). Mais que faire si nous voulons comparer **plusieurs variables** sur le même graphique ?

Imaginons que nous voulions comparer les thermomètres Harris et Trump directement. Nos données sont actuellement en **format large** (deux colonnes séparées), mais Altair préfère souvent le **format long** (une seule colonne de valeurs).

**Concept :** La fonction `melt()` de pandas permet de "fondre" un DataFrame du format large (colonnes côte à côte) au format long (lignes empilées). C'est l'opération inverse de `pivot()`.


In [ ]:
# Exemple : Comparer les thermomètres Harris et Trump par affiliation politique

# Données en format large (2 colonnes)
df_thermo = df_clean.groupby("party_id", as_index=False)[["thermo_harris", "thermo_trump"]].mean()
print("=== Format large (wide data) ===")
print(df_thermo)
print()

# Transformation en format long avec melt()
df_thermo_long = df_thermo.melt(
    id_vars="party_id",           # Colonnes à conserver (ici, le parti)
    value_vars=["thermo_harris", "thermo_trump"],  # Colonnes à "fondre"
    var_name="candidat",           # Nom de la nouvelle colonne pour les noms de variables
    value_name="thermometre"       # Nom de la nouvelle colonne pour les valeurs
)

print("=== Format long (long data) ===")
print(df_thermo_long)


In [ ]:
# Visualisation avec les données en format long
df_thermo_long["candidat"] = df_thermo_long["candidat"].replace({
    "thermo_harris": "Harris",
    "thermo_trump": "Trump"
})

party_colors = alt.Scale(
    domain=["Harris", "Trump"],
    range=["#2E74C0", "#CB454A"]
)

chart_melt = alt.Chart(df_thermo_long).mark_bar().encode(
    x=alt.X("candidat", type="nominal", title="Candidat"),
    y=alt.Y("thermometre", type="quantitative",
            title="Thermomètre",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("candidat", type="nominal",
                   title="Candidat",
                   scale=party_colors),
    column=alt.Column("party_id", type="nominal", title="Affiliation politique")
).properties(
    title=alt.TitleParams(
        text="Évaluation des candidats selon l'affiliation politique",
        subtitle=["Les indépendants semblent évaluer Harris plus positivement.", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=150,
    height=200
).configure_axis(labelAngle=0)

chart_melt


Ce graphique en barres groupées montre les évaluations des deux candidats pour chaque affiliation politique. Les colonnes séparent les trois partis, et les barres de couleurs différentes représentent chaque candidat.

La structure en format long avec `melt()` permet de créer facilement des graphiques comparatifs. Nous retrouvons le pattern classique de polarisation affective : chaque parti évalue positivement son candidat préféré et négativement l'autre (plus nuancé pour les `indépendants`)

**Pourquoi melt() ?** Altair fonctionne de manière plus intuitive avec des données en format long : chaque ligne représente une observation, et chaque colonne une variable. `melt()` restructure vos données pour faciliter cette représentation.



## 7. Formatage des axes et utilisation des F-strings

Pour afficher des proportions ou des valeurs dynamiques dans les titres, nous pouvons utiliser le formatage d'axes et les `f-strings` (formatted strings) de Python.

**Concept :** `axis=alt.Axis(format='%')` convertit une valeur comme `0.35` en `35%`. Les `f-strings` (ex: `f"{variable}"`) permettent d'insérer dynamiquement des variables dans des strings comme des titre par exemple.


In [ ]:
# Calcul des proportions par vote
total_respondents = len(df_clean)
df_vote_counts = df_clean["vote_int_label"].value_counts(normalize=True).reset_index()
df_vote_counts.columns = ["vote_int_label", "proportion"]


In [ ]:
chart_format = alt.Chart(df_vote_counts).mark_bar().encode(
    y=alt.Y("proportion", type="quantitative",
            title="Proportion",
            axis=alt.Axis(format='%'),
            scale=alt.Scale(domain=[0, 1])),
    x=alt.X("vote_int_label", type="nominal", title=None),
    color=alt.Color("vote_int_label", type="nominal",
                   title="Intention de vote",
                   scale=vote_colors,
                   legend=None)
).properties(
    title=alt.TitleParams(
        text="Distribution de l'intention de vote",
        subtitle=[f"Sur un total de {total_respondents} répondants.", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=200,
    height=200,
).configure_axis(labelAngle=0)

chart_format


Le nombre total de répondant·es ({total_respondents}) est inséré dynamiquement dans le sous-titre via une f-string, ce qui rend le graphique automatiquement mis à jour si les données changent.

**Attention:** Si vous représentez un pourcentage, il est important que votre nombre d'observations (N=...) apparaisse quelque part sur votre graphique.


## 8. Annotations et lignes de référence

Pour faciliter la lecture des valeurs exactes, nous pouvons ajouter des annotations directement sur le graphique. Nous pouvons aussi ajouter des **lignes de référence** pour marquer des seuils ou des moyennes.

**Concept :** Nous utilisons `mark_text()` pour ajouter du texte et `mark_rule()` pour ajouter des lignes horizontales ou verticales de référence.


In [ ]:
# Nous allons réutiliser l'objet df_means contenant la polarisation affective

# Calcul de la moyenne générale pour la ligne de référence
mean_aff_pol = df_clean["aff_pol"].mean().round(1)

# 1. Couche des points
points = alt.Chart(df_means).mark_point(size=250, filled=True).encode(
    x=alt.X("pol_int_label", type="ordinal",
            title="Intérêt politique",
            sort=pol_int_order),
    y=alt.Y("aff_pol", type="quantitative",
            title="Polarisation affective",
            scale=alt.Scale(domain=[0, 100])),
    color=alt.Color("vote_int_label", type="nominal",
                   title="Vote",
                   scale=vote_colors
                   )
)

# 3. Couche des annotations textuelles
text_layer = alt.Chart(df_means).mark_text(
    align="center",
    baseline="bottom",
    dy=-10,
    fontSize=10,
    fontWeight="bold",
    color="black"
).encode(
    x=alt.X("pol_int_label", type="ordinal", sort=pol_int_order),
    y=alt.Y("aff_pol", type="quantitative"),
    text=alt.Text("aff_pol", type="quantitative")
)

# 4. Ligne de référence (moyenne)
mean_line = alt.Chart(pd.DataFrame({
    "aff_pol": [mean_aff_pol]
})).mark_rule(
    color="gray",
    strokeDash=[4, 4],
    strokeWidth=1.5
).encode(
    y=alt.Y("aff_pol", type="quantitative", title="Polarisation affective")
)

# Combinaison des couches
chart_annotated = (points + text_layer + mean_line).properties(
    title=alt.TitleParams(
        text="Polarisation affective selon l'intérêt et le vote",
        subtitle=[f"La ligne pointillée grise indique la moyenne générale ({mean_aff_pol}).", "Source : ANES 2024 Time Series Study"],
        anchor="start"
    ),
    width=450,
    height=300
).configure_axis(
    labelFontSize=12, titleFontSize=12, labelAngle=0
)

chart_annotated


Les annotations textuelles permettent de faciliter la lecture des valeurs précises. La ligne de référence permet de situer chaque groupe par rapport à la moyenne générale.

### Hack-Time
- Modifiez `dy=-10` en `dy=10` et observez où se déplace le texte.
- Changez `strokeDash=[4, 4]` en `strokeDash=[2, 2]` pour une ligne plus serrée.
- Supprimez temporairement la couche `text_layer` ou `mean_line` pour voir le graphique sans ces éléments.


In [ ]:
# Hack-Time : testez les annotations et lignes de référence



## 9. Utiliser les Thèmes Altair

Plutôt que de régler manuellement chaque aspect du style, Altair propose des thèmes pré-définis qui changent l'apparence globale du graphique.

**Concept :** `alt.themes.enable('nom_du_theme')` applique un style prédéfini. Altair propose plusieurs thèmes intégrés comme `fivethirtyeight` (style journalistique) ou `dark`.

**Pour explorer tous les thèmes disponibles :** [Altair Themes Documentation](https://altair-viz.github.io/user_guide/customization.html#built-in-themes)


In [ ]:
# Activer le thème "fivethirtyeight" (style journalistique)
alt.themes.enable('fivethirtyeight')

# Reprenons le graphique précédent
chart_annotated


C'est identique au graphique précédent, mais avec un style différent.

Les thèmes sont utiles pour produire rapidement des graphiques avec un style plus pro?

Le thème "FiveThirtyEight", dans cet exemple, donne un look plus épuré. 

Pour revenir au style par défaut d'Altair :


In [ ]:
alt.themes.enable('default')


## 10. Exporter vos figures

Pour votre infographie finale ou pour un mémoire, vous avez besoin de sauvegarder l'image sur votre ordinateur. Chaque format a ses avantages.

**Concepts des formats :**

| Format | Avantages | Inconvénients | Usage recommandé |
|--------|-----------|---------------|-------------------|
| **PNG** | Compatible partout, perte de qualité minime au redimensionnement | Ne supporte pas l'interactivité | Documents Word, présentations, réseaux sociaux |
| **PDF** | Vectoriel (qualité infinie), supporte le transparent, idéal pour l'impression | Peut être lourd, pas d'interactivité | Publications académiques, rapports professionnels, impression |
| **SVG** | Vectoriel, léger, éditable dans Illustrator/Inkscape | Pas de rendu natif dans Word/PowerPoint | Développement web, graphiques à modifier |

**Note :** Le paramètre `scale_factor=2.0` double la résolution de l'image (utile pour des formats comme le PNG).

### Hack-Time
- Décommentez les lignes ci-dessus pour exporter votre figure.


In [ ]:
# Sauvegarde du dernier graphique annoté

# Format PNG (image matricielle, idéal pour Word/PowerPoint)
# Décommentez la ligne ci-dessous pour essayer :
# chart_annotated.save("polarisation_politique.png", scale_factor=2.0)

# Format PDF (vectoriel, idéal pour la publication académique)
# chart_annotated.save("polarisation_politique.pdf")

# Format SVG (vectoriel, idéal pour le web ou modification graphique)
# chart_annotated.save("polarisation_politique.svg")


Lorsque vous générez une figure avec Altair dans Google Colab, vous pouvez l'enregistrer en utilisant la méthode `.save()`. Par défaut, le fichier est sauvegardé dans l’environnement de travail de Colab, généralement dans le répertoire `/content/`.

Pour retrouver votre figure :

* Ouvrez le panneau de fichiers (icône dossier à gauche dans Colab, celle qui ressemble à un dossier, la 6ème ?)
* Regardez les contenu de ce dossier.
* Votre fichier (par exemple `polarisation_politique.png`, `polarisation_politique.pdf` ou `polarisation_politique.svg`) s'y trouve

Vous pouvez ensuite le télécharger directement depuis ce panneau.

Attention : les fichiers enregistrés dans Colab sont temporaires et seront supprimés à la fin de la session si vous ne les téléchargez pas. 

**MAIS** vous avez maintenant votre code qui vous permet de reproduire vos graphiques à l'infini 🚀
